In [ ]:
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"]="0"

import numpy as np
import tensorflow as tf
from tabulate import tabulate
from tensorflow.python.client import device_lib
from tqdm import tqdm
import time

In [ ]:

def print_gpu_details():
    """Prints all available GPUs in a formatted table with key details"""
    # Get list of all devices
    devices = device_lib.list_local_devices()
    gpu_details = []
    
    for device in devices:
        if device.device_type == 'GPU':
            # Velocity_Vaultct details from the device description string
            desc = device.physical_device_desc
            details = {
                'Device ID': device.name.split(':')[-1],
                'Name': desc.split('name: ')[1].split(',')[0] if 'name: ' in desc else 'Unknown',
                'Memory (GB)': f"{device.memory_limit / (1024**3):.2f}",
                'PCI Bus ID': desc.split('pci bus id: ')[1].split(',')[0] if 'pci bus id: ' in desc else 'Unknown',
                'GFX Version': os.environ.get('HSA_OVERRIDE_GFX_VERSION', 'Native')
            }
            gpu_details.append(details)
    
    # Print table if GPUs found
    if gpu_details:
        print("\n" + "="*85)
        print("ACTIVE GPU CONFIGURATION".center(85))
        print("="*85)
        print(tabulate(gpu_details, headers="keys", tablefmt="grid"))
        print("="*85+ "\n")
    else:
        print("No GPU devices found!")


def configure_gpu(vram_limit):
    """Configure GPU settings with optional GFX override"""
    
    # Verify GPU availability
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        try:
            # Limit VRAM on the specified GPU
            
            for gid in range(len(gpus)):
                
                tf.config.experimental.set_virtual_device_configuration(
                    gpus[gid],
                    [tf.config.experimental.VirtualDeviceConfiguration(
                        memory_limit=vram_limit[gid] * 1024)]  # Convert GB to MB
                )
                print(f"GPU {gid} VRAM limited to {vram_limit[gid]}GB")
                
        except RuntimeError as e:
            print(f"Error setting VRAM limit: {e}")
    
    if not gpus:
        raise RuntimeError(f"No GPU found")
    
    for gid in range(len(gpus)):
        print(f"Configured GPU {gid}: {tf.config.experimental.get_device_details(gpus[gid])}") 
        
        with tf.device('/GPU:'+str(gid)):  # Force GPU usage
            x = tf.ones((1, 1))    # Smallest possible tensor
            y = x + 1              # Simple operation
            y.numpy()              # Force execution

In [ ]:
VRAM = [3.0]

configure_gpu(VRAM)

print_gpu_details()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models # type: ignore
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import Sequence # type: ignore
from tensorflow.keras.callbacks import ModelCheckpoint # type: ignore

# -------------------------
# Constants
# -------------------------

dataset_detail=np.load("/mnt/Velocity_Vault/Project_Storage/stereo_ML_dataset/org/"+"detail.npy", allow_pickle=True)
dataset_detail=dataset_detail.tolist()

patch_shape = (dataset_detail[0],dataset_detail[1])

target = int(dataset_detail[2]*0.2)

Dmax = dataset_detail[3]

resize_fraction = dataset_detail[4]
resize_factor = 1/resize_fraction

BATCH_SIZE = 512
EPOCHS = 100



In [ ]:

org_path = "/mnt/Velocity_Vault/Project_Storage/stereo_ML_dataset/"
dataset_org_path = "/mnt/Velocity_Vault/Project_Storage/stereo_ML_dataset/test/"

# -------------------------
# Load memmap datasets
# -------------------------
left_patch_memmap = dataset_org_path + "left_patch.dat"
right_strip_memmap = dataset_org_path + "right_strip.dat"
patch_disparity_memmap = dataset_org_path + "patch_disp.dat"

model_location = org_path+"model/"

left_patches  = np.memmap(left_patch_memmap, dtype=np.uint8, mode='r', shape=(target, patch_shape[0], patch_shape[1]))
right_strips  = np.memmap(right_strip_memmap, dtype=np.uint8, mode='r', shape=(target, patch_shape[0], patch_shape[1] + Dmax))
median_disp   = np.memmap(patch_disparity_memmap, dtype=np.int16, mode='r', shape=(target,))





In [ ]:
from tensorflow.keras.utils import Sequence
import numpy as np
from sklearn.model_selection import train_test_split

# -------------------------
# Memmap data generator (uses patch_shape)
# -------------------------
class MemmapGenerator(Sequence):
    def __init__(self, left_mm, right_mm, disp_mm, indices,
                 batch_size=8, patch_shape=(0, 0), Dmax=800):
        self.left_mm = left_mm
        self.right_mm = right_mm
        self.disp_mm = disp_mm
        self.indices = indices
        self.batch_size = batch_size
        self.patch_shape = patch_shape
        self.Dmax = Dmax

    def __len__(self):
        return len(self.indices) // self.batch_size

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        left_batch = self.left_mm[batch_idx]
        right_batch = self.right_mm[batch_idx]
        disp_batch = self.disp_mm[batch_idx].astype('float32') / self.Dmax

        h, w = self.patch_shape
        Xl = np.expand_dims(left_batch[:, :h, :w], axis=-1) / 255.0
        Xr = np.expand_dims(right_batch[:, :h, :w + self.Dmax], axis=-1) / 255.0

        return (Xl.astype('float32'), Xr.astype('float32')), disp_batch


# -------------------------
# Create training and validation generators
# -------------------------
# Use the full dataset
idx_test = np.arange(target)

test_gen = MemmapGenerator(
    left_patches, right_strips, median_disp,
    idx_test, batch_size=BATCH_SIZE,
    patch_shape=patch_shape, Dmax=Dmax
)


In [ ]:
# -------------------------
# LOAD TRAINED MODEL
# -------------------------
model_path = model_location+"best_model.keras"  # or model_final.keras
model = tf.keras.models.load_model(model_path)
print(f"Loaded model from {model_path}")

# model.summary()

# Load
loaded_history = np.load(model_location+"history.npy",allow_pickle=True).item()
print(loaded_history.keys())   # e.g. dict_keys(['loss', 'mse', 'val_loss', 'val_mse'])

import matplotlib.pyplot as plt

for key in loaded_history:

    plt.plot(loaded_history[key], label=key)
    
plt.legend()
plt.show()


In [ ]:
# -------------------------
# EVALUATE
# -------------------------
test_loss, test_mae, test_mse = model.evaluate(test_gen, verbose=1)
print(f"Test Loss (Huber): {test_loss:.4f} | Test MAE: {test_mae:.4f} | Test MSE: {test_mse:.4f}")


In [ ]:
# -------------------------
# PREDICT
# -------------------------

preds = model.predict(test_gen, steps=1000).flatten() * Dmax
print("Predicted disparities (first 5):", preds[:5])
print("Predicted disparities (first 5):", median_disp[:5])